# Chroma DB

Chroma DB is an open source vector database. When you create embeddings with your embedding model, you can store those embeddings in ChromaDB.
Chroma DB has 3 operating options:

1. Store embeddings locally
2. Store embeddings on Chroma's hosted database
3. Use third party/host your own cloud hosted database

For this walkthrough, we'll use Chroma DB locally. Check Appendix A to see how to use Chroma's hosted database.


First, install chromadb. I'm using uv, so I'll do ```uv add chromadb```

In [12]:
!uv add chromadb

Resolved 158 packages in 0.96ms
Audited 153 packages in 0.07ms


In [13]:
import chromadb

If you're using python, chroma will handle all the backend tasks like creating a server in the background.
To use chroma DB, first connect to a chroma client

Before moving forward, I'll import some packages that we'll use during this walkthrough, I'll explain their use throughout the walkthrough.

In [14]:
import os

In [15]:
client = chromadb.PersistentClient()

Notice how we've used PersistentClient, not Client. This is because if you use Client, whatever data you add to the databse will be erased once your program completes execution. Persistent client on the other hand, creates a permenant database that you can use across executions. 

You'll find this especially useful because creating embeddings can sometimes take a long time, and doing that over and over for every execution adds up that time.

## ChromaDB Quickstart

For this project, we'll create a Docker Docs assistant which'll answer our questions from the Docker official documentation. You can find Docker documentation [here](https://github.com/docker/docs). 

First, let's clone the Docker docs from their Github repository.

In [10]:
!git clone https://www.github.com/docker/docs

Cloning into 'docs'...
remote: Enumerating objects: 470413, done.
remote: Counting objects: 100% (608/608), done.
remote: Compressing objects: 100% (326/326), done.
remote: Total 470413 (delta 428), reused 282 (delta 282), pack-reused 469805 (from 3)
Receiving objects: 100% (470413/470413), 715.96 MiB | 15.83 MiB/s, done.
Resolving deltas: 100% (303387/303387), done.


Now we have docs in ./docs
If you see, you'll find that there are may .md files in the directory. These will be our documents on which we'll perform RAG.


Let's first create a ChromaDB collection. As the name suggests, a collection is a place where you store your documents. A single ChromaDB client can have many collections. 

In [13]:
collection = client.create_collection("docker_docs")

Now that we have a collection, let's add some documents to it. Here's an example:

In [15]:
collection.add(["document-1"], documents=["Hi, I am learning RAG."])

If you run this on your own machine, chromadb will first download the "all-MiniLM-L6-v2" embedding model. Then it'll embed all the documents in the documents list provided as function's named argument "documents" and store it with the corresponding id from the ids list provided as the function's positional argument.

Few important things to take note of:

1. You pass the documents as a list, this means that you can pass multiple documents at once.
2. If the length of id's list is different than the length of the documents list, you'll get an error.
3. If you pass a document with an id for which a document is already present in the collection, it'll be overwritten with the new document.

Below is an illustration of note-2.

In [17]:
collection.add(["doc-1", "doc-2"], documents=["Document 1's contents", "Document 2's contents", "Document 3's contents"])


ValueError: Unequal lengths for fields: ids: 2, documents: 3 in add.

Let's add a few more documents in the db.

In [20]:
collection.add(
    ["id-1", "id-2", "id-3", "id-4", "id-5", "id-6"],
    documents=[
        "Docker is Docker is an open-source platform that enables developers to build, ship, and run applications within containers",
        "ChromaDB is an open-source, lightweight vector database designed to store, manage, and query vector embeddings for AI applications",
        "Linux is a free, open-source operating system (OS) based on Unix",
        "PyTorch is a popular open-source machine learning framework mainly used for building and training deep neural networks",
        "Solomon Hykes is the creator of Docker.",
        "Docker automates the deployment of applications within lightweight containers, enabling them to run consistently across different computing environments."
    ])
        

Now let's query over these documents

In [22]:
collection.query(query_texts=["What's docker?"])

{'ids': [['id-1', 'id-6', 'id-5', 'id-3', 'id-4', 'id-2', 'document-1']],
 'embeddings': None,
 'documents': [['Docker is Docker is an open-source platform that enables developers to build, ship, and run applications within containers',
   'Docker automates the deployment of applications within lightweight containers, enabling them to run consistently across different computing environments.',
   'Solomon Hykes is the creator of Docker.',
   'Linux is a free, open-source operating system (OS) based on Unix',
   'PyTorch is a popular open-source machine learning framework mainly used for building and training deep neural networks',
   'ChromaDB is an open-source, lightweight vector database designed to store, manage, and query vector embeddings for AI applications',
   'Hi, I am learning RAG.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None, None, None, None, None]],
 'distances': [[0.3558974266052246,
   0.70066225

As you can see, the most important (most close) documents to the query are present to the top and others follow in descending order of relevance. Another thing to note here is that we can pass multiple querie texts at once, and we'll get the combined rankings list.

## Creating a Docker Docs assistant 

Let's now create a Docker Docs assistant. First we'll need to delete the existing collection that we created. Or, you can create a new collection with a different name instead.

In [28]:
client.delete_collection("docker_docs")

Now let's create it again to use in our project

In [29]:
collection = client.create_collection("docker_docs")

Now we'll add all the md files present in the ./docs directory to documents. We'll give them id as "path/to/the/document".

In [38]:
for rootdirpath, dirnames, filenames in os.walk("./docs"):
    for filename in filenames:
        if filename.endswith(".md"):
            filepath = os.path.join(rootdirpath, filename)
            with open(filepath, "r") as f:
                collection.add([filepath], documents=[f.read()])

Now let's query over the collection

In [39]:
results = collection.query(query_texts=["How do I run docker in terminal"], n_results=1) # n_results limits the maximum number of results returned

In [40]:
print(results)

{'ids': [['./docs/content/includes/open-terminal.md']], 'embeddings': None, 'documents': [['> [!TIP]\n>\n> To run Docker commands, you must use a terminal. Based on your\n> operating system, you can open a terminal by doing the following:\n>\n> For Windows, select the Start Menu, specify `cmd`, and then select\n> **Command Prompt**.\n>\n> For Mac, select the **Launchpad** icon in the Dock, specify `Terminal` in the\n> search field, then select **Terminal**.\n']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[0.37393897771835327]]}


This is great! We can now just pass this to an LLM and ask it questions based on it.

Of course, most LLMs know a lot about Docker, but imagine this on your company's data, or custom files that you want to query. Furthermore, small language models may not know Docker docs or any other general information, RAG greatly enhances their capability.

## Making API calls

Now that we've the ability to retrieve information from the context, it's time to pipe that info to an LLM and ask it questions. For this one, I'll add some specific contents in a new collection, as our LLM already knows a lot about Docker.

In [55]:
from anthropic import Anthropic
from dotenv import load_dotenv
import re

In [17]:
load_dotenv()

True

In [43]:
print(os.environ.get("HI"))

hi


In [47]:
test_collection = client.get_collection("test_collection")
for rootdirpath, dirnames, filenames in os.walk("./random_stories"):
    for filename in filenames:
        if filename.endswith(".md"):
            filepath = os.path.join(rootdirpath, filename)
            with open(filepath, "r") as f:
                test_collection.add([filepath], documents=[f.read()])

results = test_collection.query(query_texts=["Summarize the Luke's story"], n_results=1) # n_results limits the maximum number of results returned
context = results["documents"][0]
anthropic_client = Anthropic()
message = anthropic_client.messages.create(
    max_tokens = 1024,
    system = f"You are an helpful AI assistant used in a RAG setting. Answer the user's question based on the provided context. The context is: \n\n\n\n\n {context}",
    messages = [
        {
            "role": "user",
            "content": "Summarize the Luke's story"
        }
    ],
    model = "claude-opus-4-7"
)
print(message.content[0].text)

# Summary of Luke's Story

**"The Crumpled Rock"** is a short, whimsical tale about Luke Blunder, an understanding and adorable squash drinker living in old-fashioned Upper Boggington, a place known for its "jealous, joyous jungle."

## Key Events:

1. **The Setup**: Luke stands by his window holding a crumpled rock, feeling puzzled, and reflecting on his charming surroundings.

2. **The Arrival**: He spots Graham Snozcumber in the distance — a gentle carer with scrawny eyelashes and curvaceous lips — approaching his home.

3. **The Confession**: As sleet rains down "like running flamingos," Graham approaches Luke and declares in hushed tones: *"I love you and I want a phone number."*

4. **The Response**: Luke, still fingering his crumpled rock, replies simply, *"Graham, d'oh."* The two share a tender moment, compared to "two racid, real rabbits jogging at a very gracious wedding."

5. **The Happy Ending**: Luke reveals he feels the same way, Graham looks relaxed and emotionally "blus

This above, is the generation part of RAG. We retrieve relevant context to the query and feed it to the LLM for generation.

After this, everything that we'll do will be about solving problems that have troubled us while doing RAG.

## Chunking

Let's go back to our docker example. Remember that the collection name there was docker_docs.

In [48]:
docker_collection = client.get_collection("docker_docs")

Let's ask a question based on this collection.

In [51]:
results = docker_collection.query(
    query_texts = ["What is docker, and how do I run it in terminal?"],
    n_results = 1
)
for document in results["documents"][0]:
    print(document)

> [!TIP]
>
> To run Docker commands, you must use a terminal. Based on your
> operating system, you can open a terminal by doing the following:
>
> For Windows, select the Start Menu, specify `cmd`, and then select
> **Command Prompt**.
>
> For Mac, select the **Launchpad** icon in the Dock, specify `Terminal` in the
> search field, then select **Terminal**.



Now, this document doesn't really contain what we ask for, so let's increase n_results to 2

In [52]:
results = docker_collection.query(
    query_texts = ["What is docker, and how do I run it in terminal?"],
    n_results = 2
)
for document in results["documents"][0]:
    print(document)

> [!TIP]
>
> To run Docker commands, you must use a terminal. Based on your
> operating system, you can open a terminal by doing the following:
>
> For Windows, select the Start Menu, specify `cmd`, and then select
> **Command Prompt**.
>
> For Mac, select the **Launchpad** icon in the Dock, specify `Terminal` in the
> search field, then select **Terminal**.

---
title: Get Docker Desktop 
keywords: concepts, container, docker desktop
description: This concept page will teach you download Docker Desktop and install it on Windows, Mac, and Linux
summary: |
  Getting Docker Desktop up and running is the first crucial step for
  developers diving into containerization, offering a seamless and
  user-friendly interface for managing Docker containers. Docker Desktop
  simplifies the process of building, sharing, and running applications in
  containers, ensuring consistency across different environments.
weight: 1
aliases:
 - /getting-started/get-docker-desktop/
---

{{< youtube-embed C2bPV

As you can see, our context just exploded in size. As your queries become more and more complex, the relevant information will be spread across more and more documents. Increasing number of documents will massively increase your context size, which creates needle in a haystack problem, and also increases token usage. To solve this issue, we use **Chunking**.

### What's Chunking?

Chunking is the process of re-structuring our documents while storing them in the database. 

We can chunk our documents in multiple ways:
1. Fixed size chunking: Here, we create a split in data every _k_ tokens/charaters. This is the most straightforward way of chunking.
2. Document based chunking: This is the chunking that we used above. Simply add each document as its own vector.
3. Semantic chunking: Chunk documents by splitting them by Paragraph/sentences, etc. There are also more advanced ways to do semantic chunking. For example, if you're creating an AI assisted coding agent, you'd want to store python files by chunking them by functions, scopes, etc. Or if you are creating an AI Legal assistant, you can split your documents based on clauses.


Here, we're working with markdown files. So let's do chunking the following way:

**Create a chunk whenever a # appears in the document, that is when a new title/header appears.**
Let's first delete our collection first and re-create it with new chunking procedure.

In [53]:
client.delete_collection("docker_docs")

In [54]:
docker_docs = client.create_collection("docker_docs")

In [64]:
for rootdir, dirnames, filenames in os.walk("./docs"):
    for filename in filenames:
        filepath = os.path.join(rootdir, filename)
        if filepath.endswith(".md"):
            with open (filepath, "r") as f:
                contents = f.read()
                parts = re.split(r"(?=^#+\s)", contents, flags=re.MULTILINE)
                for part in parts:
                    part.strip()
                    if part == "":
                        continue
                    docker_docs.add([f"{filename}-{part.splitlines()[0]}"], documents=[part])


Now let's query over DB and see the results. I've set n_results = 5 here.

In [67]:
results = docker_docs.query(
    query_texts = ["What is docker, and how do I run it in terminal?"],
    n_results = 5
)

for result in results["documents"][0]:
    print(result)

### Docker Desktop

Docker Desktop is an easy-to-install application for your Mac, Windows, or Linux environment that enables you to build and share containerized applications and microservices. Docker Desktop includes the Docker daemon (`dockerd`), the Docker client (`docker`), Docker Compose, Docker Content Trust, Kubernetes, and Credential Helper. For more information, see [Docker Desktop](/manuals/desktop/_index.md).


---
description: "Running and configuring containers with the Docker CLI"
keywords: "docker, run, cli"
aliases:
- /reference/run/
title: Running containers
---

Docker runs processes in isolated containers. A container is a process
which runs on a host. The host may be local or remote. When you
execute `docker run`, the container process that runs is isolated in
that it has its own file system, its own networking, and its own
isolated process tree separate from the host.

This page details how to use the `docker run` command to run containers.


> [!TIP]
>
> To run D

These are much better results with the same overall size of prompt. 